## **DESops Tutorial 02**

This guide will cover some more particular ways to use the DESops library.

In [1]:
import DESops as d

## **Outline**
* [Other Supervisory Control Methods](#Other-Supervisory-Control-Methods)
* [Language Equivalence](#Language-Equivalence)
* [Generating Random Automata](#Generating-Random-Automata)
* [Unobserverable Reach and the UR Class](#Unobserverable-Reach-and-the-UR-Class)
* [Example: Opacity](#Example:-Opacity)
* [Example: Sensor Deception Attacks](#Example:-Sensor-Deception-Attacks)

___

### **Other Supervisory Control Methods**

There are additional algorithms for solving supervisory control problems implemented in `DESops` in the `supervisor` submodule.

Namely, there is an offline implementation of the [VLPPO algorithm](https://link.springer.com/content/pdf/10.1007/BF01797138.pdf) and an implementation of the [All Enforcement Structure](https://ieeexplore.ieee.org/document/7286769) (AES) method.

These functions use the same mostly the same interface as the supremal sublanguage functions. The input is generally `method(plant, spec, [...])` where [...] are optional parameters. `plant` should be a `DFA`, but `spec` can be either a `DFA` (automata/language based specification) *or* a `set()` of names of states in `plant` (state based specification).

However, the information returned by the methods is not uniform. In the case of `supervisor.supremal_sublangauge`, a realization of a supervisor $S$ is implicitly computed, but the `DFA` returned is $S  ||  G$.

`supervisor.offline_VLPPO` on the other hand returns a `DFA` which is a realization of the supervisor $S$.

Computing a supervisor using the AES methods first requires constructing an arena using `supervisor.construct_arena`, which returns a tuple (`DFA`, `DFA`) where the first item is the AES arena after pruning state violations, and the second item is the complete arena. Then the supervisor must be extracted from the AES arena using `supervisor.extract_arena`.

Finally, both the AES and VLPPO methods as implemented in `DESops` will not consider the marked case, *i.e. they always compute the prefix-closed supervisors*. Only `supervisor.supremal_sublangauge` with argument `prefix_closed=False` considers the marked case.

In [2]:
G = d.read_fsm("tests/models/vlppo_tests/plant1.fsm")
print(G)

# Create a state based specification, where state "5" in G is unsafe:
X_crit = set()
X_crit.add("5")

DFA : 5 V, 7 E
Events: {c, a, b}
Euc: {a}
Euo: {c}
Source | (Target, Event), ...)
1  :  [('2', a), ('3', b)]
2  :  [('5', c), ('4', b)]
3  :  [('4', a)]
4  :  [('1', c), ('1', a)]
5  :  []



In [3]:
S_vlppo = d.supervisor.offline_VLPPO(G, X_crit, event_ordering=["b", "c"])
S_vlppo_par_G = d.composition.parallel(S_vlppo, G)
print(S_vlppo_par_G)

DFA : 13 V, 20 E
Events: {c, b, a}
Euc: {a}
Euo: frozenset({c})
Source | (Target, Event), ...)
('0', '1')  :  [(('1', '3'), b), (('5', '2'), a)]
('1', '3')  :  [(('3', '4'), a)]
('5', '2')  :  [(('3', '4'), b)]
('3', '4')  :  [(('3', '1'), c), (('4', '1'), a)]
('3', '1')  :  [(('1', '3'), b), (('4', '2'), a)]
('4', '1')  :  [(('5', '2'), a), (('2', '3'), b)]
('4', '2')  :  [(('2', '4'), b)]
('2', '3')  :  [(('6', '4'), a)]
('2', '4')  :  [(('2', '1'), c), (('6', '1'), a)]
('6', '4')  :  [(('4', '1'), a)]
('2', '1')  :  [(('1', '3'), b), (('6', '2'), a)]
('6', '1')  :  [(('4', '2'), a), (('2', '3'), b)]
('6', '2')  :  [(('2', '4'), b)]



In [4]:
AES_arena, total_arena = d.supervisor.construct_AES(G, X_crit)
print("# states pruned from total_arena: ", total_arena.vcount() - AES_arena.vcount())

# states pruned from total_arena:  6


In [5]:
S_aes = d.supervisor.extract_AES_super(AES_arena)
S_aes_par_G = d.composition.parallel(S_aes, G)
print(S_aes_par_G)

DFA : 11 V, 16 E
Events: {c, b, a}
Euc: {a}
Euo: {c}
Source | (Target, Event), ...)
('0', '1')  :  [(('2', '3'), b), (('1', '2'), a)]
('2', '3')  :  [(('3', '4'), a)]
('1', '2')  :  [(('3', '4'), b)]
('3', '4')  :  [(('3', '1'), c), (('0', '1'), a)]
('3', '1')  :  [(('2', '3'), b), (('0', '2'), a)]
('0', '2')  :  [(('0', '5'), c), (('2', '4'), b)]
('0', '5')  :  []
('2', '4')  :  [(('2', '1'), c), (('3', '1'), a)]
('2', '1')  :  [(('2', '3'), b), (('3', '2'), a)]
('3', '2')  :  [(('3', '5'), c), (('2', '4'), b)]
('3', '5')  :  []



In [6]:
S_scn = d.supervisor.supremal_sublanguage(G, X_crit, prefix_closed=True)
print(S_scn)

DFA : 0 V, 0 E
Events: set()
Euc: set()
Euo: set()
Source | (Target, Event), ...)



Note that the supremal controllable and normal sublanguage does not exist in this case. AES and VLPPO both compute supervisors which generate incomparable maximal controllable and observable languages (in this case, they occassionally do compute the same maximal language). 

In the special case where $E_{c} \subseteq E_{o}$, all three methods will compute a supervisor with the supremal controllable and observable language. We demonstrate this property using a function in `DESops` to verify language equivalence in the next section.

___

### **Language Equivalence**

As an equality check of DFAs, the `compare_language(input1, input2)` function returns whether the input DFAs are language equivalent.


For example, the supervisors in the previous section are not necessarily language equivalent:

In [7]:
# True: computed the same maximal solution. Notice how both S_vlppo_par_G and S_aes_par_G have 13 vertices
# False: computed incomparable maximal solutions. S_vlppo_par_G will have 13 vertices, and S_aes_par_G 11 vertices
d.compare_language(S_vlppo_par_G, S_aes_par_G)

False

In the special case $E_{c} \subseteq E_{o}$, where normality implies observability, we should expect all three supervisory control methods to compute the same supremal controllable and observable supervisor. We will verify this with another example using `compare_language`.

In [24]:
G_po = d.read_fsm("tests/models/scn_tests/cn_test4_g.fsm")
H_po = d.read_fsm("tests/models/scn_tests/cn_test4_h.fsm")

Euc = G_po.Euc.union(H_po.Euc)
Euo = G_po.Euo.union(H_po.Euo)
print("Uncontrollable: ", Euc)
print("Unobservable: ", Euo)

# Find the contrapositive, Euo C Euc ==> Ec C Eo
all_uo_are_uc = all(e in Euc for e in Euc)
print("All controllable events observable? ", all_uo_are_uc)

Uncontrollable:  {c, d}
Unobservable:  {d}
All controllable events observable?  True


In [25]:
S_scn = d.supervisor.supremal_sublanguage(G_po, H_po, mode=d.supervisor.Mode.CONTROLLABLE_NORMAL, prefix_closed=True)
print(S_scn)

DFA : 2 V, 1 E
Events: {c, a, d, b}
Euc: {c, d}
Euo: {d}
Source | (Target, Event), ...)
0  :  [('2', b)]
2  :  []



In [26]:
V = d.supervisor.offline_VLPPO(G_po, H_po)
print(V)

DFA : 2 V, 6 E
Events: {c, d, a, b}
Euc: {c, d}
Euo: frozenset({d})
Source | (Target, Event), ...)
0  :  [('0', c), ('0', d), ('1', b)]
1  :  [('1', c), ('1', d), ('1', b)]



In [27]:
# Compose the supervisor realization with the plant to compare with supervisor.supremal_sublanguage:
V_scn = d.composition.parallel(V, G_po)
print(V_scn)

DFA : 2 V, 1 E
Events: {c, d, b, a}
Euc: {c, d}
Euo: frozenset({d})
Source | (Target, Event), ...)
('0', '0')  :  [(('1', '2'), b)]
('1', '2')  :  []



In [28]:
# Verify that the languages are equivalent:
print(d.compare_language(S_scn, V_scn))
# d.compare_language is equivalent to '=='' :
print(S_scn == V_scn)

True
True


And the supervisor computed by AES will also be equivalent:

In [69]:
AES, T = d.supervisor.construct_AES(G_po, H_po)
A = d.supervisor.extract_AES_super(AES)
A_scn = d.composition.parallel(A, G_po)

V_scn == A_scn == S_scn

True

___

### **Generating Random Automata**

Generating random automata is a useful way to approach testing and benchmarking. `DESops` has two methods for generating random automata included in the `randoma_automata` submodule.

* `generate(num_states, num_events[, min_trans_per_state=0, max_trans_per_state=None, det=True, num_init=1, num_marked=0, num_secret=0, num_uo=0, num_uc=0, filename=None, enforce_ccessibility=True, enforce_max_trans_per_state=True])`

* `generate_regal(num_states, num_events[, num_uo=0, num_uc=0, g=None, timeout=None, max_out_degree=None, max_parallel_edges=None, remove_self_loop_prob=0])`

`generate` uses a (pseudo)random method to generate either random `DFA` or `NFA` objects.

`generate_regal` uses the [REGAL c++ library](https://link.springer.com/chapter/10.1007/978-3-540-76336-9_28) to generate random automata. However, using this method requires an additional installation step. See the installation readme in `DESops/random_automata/regal_readme.txt` for installation instructions. Using the `generate_regal` function without properly installing REGAL will result in a `DependencyNotInstalledError`.

We demonstrate using the `generate` function, and use it to verify that our supervisory control algorithms all produce the same result.

In [4]:
V = 10   # num states
E = 4    # num events
#G_rand = d.random_automata.generate_regal(V, E, num_uo=1)
G_rand = d.random_automata.generate(V, E, num_uo=1)
G_rand.Euc = G_rand.Euo

print(G_rand)

X_crit = set()

# The last state in G_rand is unsafe:
X_crit.add(G_rand.vs["name"][-1])
print("Critical state: ", X_crit)

DFA : 9 V, 43 E
Events: {0, 1, 3, 2, 4}
Euc: set()
Euo: {4}
Source | (Target, Event), ...)
0  :  [('0', 1), ('1', 2), ('8', 0), ('8', 4), ('6', 3)]
1  :  [('2', 3), ('8', 2), ('6', 1), ('4', 4)]
2  :  [('0', 4), ('1', 1), ('3', 3), ('3', 0), ('2', 2)]
3  :  [('4', 1), ('4', 3), ('5', 0), ('5', 2), ('5', 4)]
4  :  [('5', 3), ('5', 4), ('0', 1), ('8', 0), ('1', 2)]
5  :  [('2', 4), ('5', 1), ('6', 0), ('7', 3)]
6  :  [('3', 1), ('1', 4), ('4', 0), ('0', 3), ('2', 2)]
7  :  [('8', 1), ('0', 3), ('3', 4), ('3', 0), ('6', 2)]
8  :  [('2', 0), ('5', 1), ('6', 4), ('3', 2), ('0', 3)]

DFA : 10 V, 22 E
Events: {a, b, d, c}
Euc: {c}
Euo: {c}
Source | (Target, Event), ...)
0  :  [('0', a), ('3', d), ('5', c), ('7', b)]
1  :  [('1', a), ('4', b), ('5', c), ('7', d)]
2  :  [('1', d), ('3', b), ('9', c)]
3  :  [('2', b), ('4', c), ('6', d), ('9', a)]
4  :  [('2', d), ('7', a)]
5  :  [('0', a), ('1', c)]
6  :  [('5', a), ('9', d)]
7  :  [('8', d)]
8  :  []
9  :  []

Critical state:  {'9'}


In [18]:
# VLPPO:
V_sup = d.supervisor.offline_VLPPO(G_rand, X_crit)
V = d.composition.parallel(V_sup, G_rand)

# AES:
AES, T = d.supervisor.construct_AES(G_rand, X_crit)
A_sup = d.supervisor.extract_AES_super(AES)
A = d.composition.parallel(A_sup, G_rand)

# sup contr norm:
S = d.supervisor.supremal_sublanguage(G_rand, X_crit, prefix_closed=True)

V == A == S

True

Random automata can also be useful for benchmarking algorithms, as a means to gauge "average" runtime (under the assumption that the random generation process is representative of some sort of average automata). Generally, VLPPO or AES will scale much more effectively than the supremal sublanguage case.

In [19]:
%%timeit 
# For a fair comparison, we include the parallel composition in the timing of VLPPO
V_sup = d.supervisor.offline_VLPPO(G_rand, X_crit)
d.composition.parallel(G_rand, V_sup)

277 µs ± 4.03 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [20]:
%%timeit
d.supervisor.supremal_sublanguage(G_rand, X_crit, prefix_closed=True)

61.7 ms ± 3.81 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


___

### **Unobserverable Reach and the UR Class**

For partially observed systems, we often want to compute a state estimate after a string of observable events.

The unobservable reach of a state $x \in X$ under a set of possible unobservable events $E_{uo}^{'} \subseteq E_{uo}$ is defined as:

$ UR(x, E_{uo}^{'}) = \{y \in X; (\exists t \in E_{uo}^{'\ *}) [f(x, t) = y]\} $

Where $f: X, E -> X$ is a transition function, and $E_{uo}^{' *}$ is the Kleene closure of the subset of unobservable events.

The unobservable reach is extended to a set of states $B \subseteq X$:

$ UR(B, E_{uo}^{'}) = \cup_{x \in B} UR(x, E_{uo}^{'})$

These subset constructions are expensive, and in some cases are computed an exponential number of times. Memoization can greatly speed up these constructions by storing the results of unobservable reach computations.

In `DESops`, the `UR` class uses a simple dictionary (no size limit) to store unobservable reach computations from sets. The cache maps $(B, E_{uo}^{'}) \rightarrow \{\emptyset, UR(B, E_{uo}^{'})\}$, and computes the unobservable reach if it isn't found (maps to $\emptyset$).

We can see that the class is used in computing the observer:

In [21]:
G_rand = d.random_automata.generate(100, 10, num_uc=2, num_uo=2)

In [22]:
G_rand_obs = d.composition.observer(G_rand)
print(G_rand.UR)

UR Class
   Sets of states: 30881
   Len(keys): 586533
   Len(items): 61762


Printing the UR class gives a rough summary of how much has been stored in the cache.

The cache can be emptied using `UR.empty_cache()`.

In [23]:
G_rand.UR.empty_cache()
print(G_rand.UR)

UR Class
   Sets of states: 0
   Len(keys): 0
   Len(items): 0


#### Computing observable reach using the UR class

We compute the unobservable reach using the method `UR.from_set(set_of_states, events=None, freeze_result=False)`

* `set_of_states` is a set of indices of vertices, $B$ above.
* `events` is the set of events to consider, $E_{uo}^{'}$ above. If it is none (default behavior), the parent Euo attribute is used.
* `freeze_result`: boolean variable, if `True` then `from_set` will store and return a `frozenset()` of states instead of a regular set. This is useful if the state estimates need to be hashable, and generally they need not be mutable.

In [24]:
state_est = G_rand.UR.from_set(frozenset([0, 1, 2]), freeze_result=True)
print(state_est)

frozenset({0, 1, 2, 4, 6, 15, 82, 19, 84, 23, 88, 90, 32, 98, 99, 38, 44, 53, 55, 63})


In [27]:
# Check the (key, val) pair in the underlying dict:
states = frozenset([0, 1, 2])

# `events` should be hashable. The class converts nonhashable types to `frozenset()`
events = frozenset(G_rand.Euo)

key = (states, events)
key in G_rand.UR.set_of_states_dict

True

___

### **Example: Opacity**
* briefly explain opacity research
* link to a paper?
* show opacity test with DESops

___

### **Example: Sensor Deception Attacks**
* briefly explain SDA research
* link to a paper 
* show SDA test with DESops (maybe robust & link to robust paper)
* stochastic example with prism